# Parameter Sweep (beta_C, gamma_C) 

In [1]:
%load_ext autoreload
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime

In [2]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [3]:
%autoreload 2
from lib.graph_factory import GraphFactory
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed
from lib.classifier import ByzClassifier
from lib.system import SystemSimulator
from lib.config import BASE_CONF, LOGFILE, NUM_NODES

In [4]:
RUN_DIR = Path().resolve()
config = BASE_CONF

In [5]:
global_dataset = fetch_dataset('MNIST')
gf = GraphFactory(config['train']['num_nodes'], config['train']['b'], seed('graph'))
proj_const_estimator = ProjConstEstimator(config, global_dataset, rng('proj-const-estimator')) 
classifier = ByzClassifier(config, global_dataset, rng('byz-classifier'))
sys_sim = SystemSimulator(config, global_dataset, rng('ml-system-sim'), LOGFILE)

In [6]:
prelim_metrics = dict()

proj_const_estimator.configure(gf,config)
proj_const = proj_const_estimator.estimate()
prelim_metrics['proj_const'] = proj_const

classifier.configure(gf,config, proj_const)
X_C_train, y_C_train, X_C_test, y_C_test = classifier.run_simulations()
C_opt_pt = classifier.train_and_eval(X_C_train, y_C_train, quantile=0.7)
C_metrics = classifier.test(X_C_test, y_C_test)
C_best_est, C_pre_pipe = classifier.get_params()
prelim_metrics.update(C_metrics)
prelim_metrics.update(C_opt_pt)

In [7]:
%%capture
fig, ax = plt.subplots(3,2, figsize=(10,12))
classifier.plot_pr(ax[2,1])

In [ ]:
sim_params = {
'algorithms':['RDSGD_ORACLE'],
'atk_type':config['sys']['atk_type'],
'threat_model':'T3'
}

In [ ]:
num_test_pts = 6
xv, yv = np.meshgrid(np.linspace(0,1,num_test_pts)[:-1], np.linspace(0,1,num_test_pts), indexing='ij')
for i in range(num_test_pts-1): # cannot allow full FPR
    for j in range(num_test_pts): 
        run_results = dict()
        RUN_ID = generate()
        run_results['timestamp'] = pd.to_datetime(datetime.now())
        run_results['proj_const'] = proj_const

        params_C = dict(C_fpr=xv[i,j], C_fnr=yv[i,j], C_tau=0) 
        run_results.update(params_C)

        sys_sim.configure(gf, config, proj_const, None, None, params_C, is_printing_logs=False)
        df= sys_sim.simulate(sim_params, run_results)
        run_results['max_test_acc'] = df.loc['RDSGD_ORACLE']['max_test_acc'].max()
        run_results['min_test_acc'] = df.loc['RDSGD_ORACLE']['min_test_acc'].min()
        run_results['term_cons'] = df.loc['RDSGD_ORACLE']['Ck'].iloc[-1]
        print(f"T3_RDSGD_ORACLE(fpr={params_C['C_fpr']:.3f}, fnr={params_C['C_fnr']:.3f})= \
              [{run_results['min_test_acc']},{run_results['max_test_acc']}]")
        payload = sys_sim.log_results(run_results, RUN_DIR, RUN_ID)

Simulating label_flip attack...
==========RDSGD==========
[k=35] train_loss=1.7283
[k=70] train_loss=1.7838
[k=105] train_loss=1.8159
===========IOS===========
[k=35] train_loss=1.5822
[k=70] train_loss=1.5780
[k=105] train_loss=1.5780
===========SCC===========
[k=35] train_loss=1.6302
[k=70] train_loss=1.6390
[k=105] train_loss=1.6419
=========TriMean=========
[k=35] train_loss=1.6674
[k=70] train_loss=1.6554
[k=105] train_loss=1.6512
==========CooMed=========
[k=35] train_loss=1.6786
[k=70] train_loss=1.6659
[k=105] train_loss=1.6619
0:25:15.192499

Simulating sign_flip attack...
==========RDSGD==========
[k=35] train_loss=1.9021
[k=70] train_loss=1.8588
[k=105] train_loss=1.7902
===========IOS===========
[k=35] train_loss=1.6381
[k=70] train_loss=1.6497
[k=105] train_loss=1.6529
===========SCC===========
[k=35] train_loss=1.6264
[k=70] train_loss=1.6255
[k=105] train_loss=1.6287
=========TriMean=========
[k=35] train_loss=1.6757
[k=70] train_loss=1.6526
[k=105] train_loss=1.6460
===

In [ ]:
plt.subplots_adjust(hspace=0.2)
fig.tight_layout()
display(fig)

In [10]:
images_dir = project_root / RUN_DIR / "images"
images_dir.mkdir(parents=True, exist_ok=True)
full_path = images_dir / "results.png"
fig.savefig(full_path, bbox_inches='tight', dpi=300)